<img src="https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/unifor-logo.png" width="400">
<br>
<b>
<font size="6" face="arial" color="blue">
    Graduação em Ciência da Computação
</font>
</b>
<br>
<b>
<font size="4" face="arial">
    Disciplina: Métodos Quantitativos em Computação
</font>
</b>

**Orientador: Prof. Me. Ricardo Carubbi** <br>
*Docente da Graduação e Pós-Graduação em Ciência de Dados e Inteligência Artificial*<br>
*Laboratório de Ciência de Dados e Inteligência Artificial*<br>
*Universidade de Fortaleza*<br>

Lattes: http://lattes.cnpq.br/5738786447903616 |
GitHub: https://github.com/carubbi/

[Unifor.br](https://unifor.br/) | [Instagram](https://www.instagram.com/uniforcomunica/?hl=pt-br) | [Facebook](https://www.facebook.com/uniforoficial/) | [Twitter](https://www.facebook.com/uniforoficial/)  | [LinkedIn](https://www.linkedin.com/school/university-of-fortaleza/?originalSubdomain=pt) | [TV Unifor](https://www.unifor.br/tv-unifor) | [G1/Ensinando e Aprendendo](https://g1.globo.com/ce/ceara/especial-publicitario/unifor/ensinando-e-aprendendo/)

# AP1 — Análise exploratória do preço de venda de imóveis residenciais

## 1. Identificação

- **Disciplina:** Métodos Quantitativos em Computação (T199)
- **Turma:** T199-64
- **Integrantes:** Adriel Medeiros Lins
- **Característica qualitativa $B$ atribuída:** Qualidade da cozinha (`Kitchen Qual`)
- **Repositório:** <https://github.com/medeiros-cs/mq-ap>

## 2. Delimitação da investigação

### 2.1 Contexto

A base reúne informações sobre imóveis residenciais, seus preços de venda e suas características. Antes de calcular probabilidades ou ajustar modelos, é necessário conhecer a estrutura, a qualidade e o comportamento dos dados.

### 2.2 Pergunta de pesquisa

Como se distribuem e variam os preços de venda de imóveis residenciais?

### 2.3 Objetivo da investigação

Caracterizar a distribuição e a variabilidade dos preços de venda e investigar suas relações com características físicas, qualitativas e temporais dos imóveis.

## 3. Carregamento dos dados

O arquivo bruto `data/raw/AmesHousing.txt`, fornecido pelo professor, será carregado em uma célula própria, preservando a cópia bruta sem alterações.

### 3.1 Importação e rastreabilidade

**Identificação e procedência do arquivo.** `AmesHousing.txt`, fornecido pelo professor Ricardo Carubbi no repositório da disciplina (`data/raw/AmesHousing.txt`). Corresponde à base *Ames Housing*, publicada por De Cock (2011) como alternativa ao conjunto *Boston Housing* para projetos de regressão. Cópia obtida em 11/09/2026 e preservada sem alterações em `data/raw/AmesHousing.txt` deste repositório.

In [ ]:
# Importar a biblioteca de manipulação de dados
import pandas as pd


#### Leitura do arquivo

In [ ]:
# Carregar o arquivo bruto fornecido pelo professor, sem aplicar transformações
df_raw = pd.read_csv("../data/raw/AmesHousing.txt", sep="\t")
df_raw.shape


**Tabela 1 - Dimensões do arquivo bruto.** O arquivo é separado por tabulação (`\t`), sem indícios de problemas de codificação (leitura padrão UTF-8 sem erros). A tabela resultante tem 2930 linhas e 82 colunas.

#### Unidade de análise

In [ ]:
# Verificar a unicidade de Order e PID para confirmar a unidade de análise
unidade_analise = pd.DataFrame({
    "Order (linhas) únicos": [df_raw["Order"].is_unique],
    "PID (imóvel) únicos": [df_raw["PID"].is_unique],
    "n_linhas": [len(df_raw)],
})
unidade_analise


**Tabela 2 - Unicidade dos identificadores.** `Order` e `PID` são ambos únicos nas 2930 linhas, confirmando que a unidade de análise é a venda individual de um imóvel residencial (um registro por transação).

#### Nomes das variáveis

In [ ]:
# Listar os nomes originais das variáveis, mantidos sem normalização nesta etapa
list(df_raw.columns)


**Nomes das variáveis.** Os nomes originais do arquivo (com espaços, ex.: `Mas Vnr Area`, `Year Remod/Add`) foram preservados sem normalização nesta etapa. Caso alguma variável específica exija um nome mais legível nas seções seguintes, o nome legível será apresentado seguido do nome original entre parênteses, conforme exigido pelo enunciado — sem alterar as colunas do `DataFrame` bruto.

### 3.2 Qualidade e preparação

#### Tipos computacionais

In [ ]:
# Verificar os tipos computacionais das colunas
df_raw.dtypes.value_counts()


**Tabela 3 - Tipos computacionais das colunas.** 43 colunas são do tipo `object` (predominantemente categóricas/texto), 28 são `int64` e 11 são `float64`. Conversões específicas serão avaliadas variável a variável, conforme necessário nas análises seguintes.

#### Duplicidades

In [ ]:
# Verificar duplicidade de linhas completas e de identificador do imóvel (PID)
pd.DataFrame({
    "linhas_duplicadas_completas": [df_raw.duplicated().sum()],
    "PID_duplicados": [df_raw["PID"].duplicated().sum()],
})


**Tabela 4 - Duplicidades.** Não há linhas completamente duplicadas nem `PID` duplicado.

#### Valores ausentes por coluna

In [ ]:
# Resumir tipo, quantidade e percentual de valores ausentes por coluna
resumo_ausentes = pd.DataFrame({
    "dtype": df_raw.dtypes,
    "n_ausentes": df_raw.isna().sum(),
    "pct_ausentes": (df_raw.isna().mean() * 100).round(1),
}).sort_values("n_ausentes", ascending=False)

resumo_ausentes.head(15)


**Tabela 5 - Valores ausentes por coluna (15 maiores).** Consultando o dicionário oficial (`data/raw/DataDocumentation.txt`), confirma-se que, em `Pool QC`, `Fence` e `Misc Feature`, o código `NA` é uma categoria válida do dicionário (respectivamente "No Pool", "No Fence" e "None"), e em `Alley`, "No alley access" — não uma ausência real, mas interpretada como tal pela leitura padrão do `pandas`. As demais colunas com poucos ausentes (ex.: `Lot Frontage`, colunas de garagem e porão) exigirão investigação específica antes de qualquer tratamento.

#### Colunas relevantes à AP2

In [ ]:
# Inspecionar tipo, ausências e intervalo das colunas relevantes à trilha Binomial da AP2
cols4 = ["Fireplaces", "Mas Vnr Area", "Year Remod/Add", "Year Built"]
inspecao_cols4 = pd.DataFrame({
    "dtype": df_raw[cols4].dtypes,
    "n_ausentes": df_raw[cols4].isna().sum(),
    "min": df_raw[cols4].min(),
    "max": df_raw[cols4].max(),
})
inspecao_cols4


**Tabela 6 - Inspeção inicial das colunas relevantes à trilha Binomial da AP2.** `Fireplaces` (`int64`, 0 a 4, sem ausências) e `Year Built`/`Year Remod/Add` (`int64`, sem ausências, intervalos plausíveis: construção entre 1872 e 2010, remodelação entre 1950 e 2010) não apresentam valores ausentes. `Mas Vnr Area` (`float64`) apresenta 23 valores ausentes, investigados a seguir.

#### Inconsistência temporal

In [ ]:
# Localizar casos em que a remodelação antecede a construção
inconsistencias_ano = df_raw[df_raw["Year Remod/Add"] < df_raw["Year Built"]]
inconsistencias_ano[["Order", "PID", "Year Built", "Year Remod/Add", "Yr Sold"]]


**Tabela 7 - Caso sinalizado de inconsistência temporal.** Uma linha (`Order` 851) registra `Year Remod/Add` (2001) anterior a `Year Built` (2002), uma inconsistência de um ano. Por afetar apenas um caso e não distorcer as análises agregadas, a linha será mantida com a inconsistência documentada; uma correção ou exclusão só será aplicada se essa observação vier a influenciar de forma relevante alguma medida ou modelo nas etapas seguintes.

#### Padrão de ausência em Mas Vnr

In [ ]:
# Comparar o padrão de ausência entre Mas Vnr Type e Mas Vnr Area
tipo_na = df_raw["Mas Vnr Type"].isna()
area_na = df_raw["Mas Vnr Area"].isna()

pd.DataFrame({
    "Tipo e área ausentes": [(tipo_na & area_na).sum()],
    "Tipo ausente, área = 0 (plausível 'sem revestimento')": [(tipo_na & ~area_na & (df_raw["Mas Vnr Area"] == 0)).sum()],
    "Tipo ausente, área > 0 (inconsistente)": [(tipo_na & ~area_na & (df_raw["Mas Vnr Area"] > 0)).sum()],
})


**Tabela 8 - Padrão de ausência em `Mas Vnr Area` e `Mas Vnr Type`.** A maior parte dos registros com `Mas Vnr Type` ausente tem `Mas Vnr Area` explicitamente igual a 0 (1745 casos), consistente com "sem revestimento em alvenaria". Os 23 registros com `Mas Vnr Area` ausente têm também `Mas Vnr Type` ausente, formando um padrão distinto dos casos de área zero — por isso, **não serão automaticamente tratados como ausência de revestimento**, conforme exigido pelo enunciado; permanecerão como valores ausentes a decidir (imputação, correção pontual ou exclusão justificada) na etapa de valores discrepantes. Adicionalmente, 7 registros têm `Mas Vnr Type` ausente com `Mas Vnr Area` maior que zero — uma inconsistência a investigar individualmente antes de qualquer tratamento.

#### Cópia de trabalho

In [ ]:
# Preservar uma cópia lógica dos dados brutos para os tratamentos das próximas etapas
df = df_raw.copy()
df.shape


**Cópia de trabalho.** A partir daqui, as etapas de tratamento e análise operam sobre `df` (cópia lógica), preservando `df_raw` intacto como registro do arquivo original.

## 4. Análise exploratória

*(Escopo do Acompanhamento 2 em diante — não iniciar antes de fechar a Seção 3.)*

### 4.1 Análise univariada

#### 4.1.1 Medidas, histogramas e gráfico de barras

In [ ]:
# TODO (Acompanhamento 2): tabela de medidas de posição, dispersão e forma de SalePrice e Gr Liv Area


In [ ]:
# TODO (Acompanhamento 2): histogramas em painel único (SalePrice e Gr Liv Area)


In [ ]:
# TODO (Acompanhamento 2): tabela de frequências absolutas/relativas e gráfico de barras de B


#### 4.1.2 Valores discrepantes (cercas de Tukey)

In [ ]:
# TODO (Acompanhamento 2): Q1, Q3, IQR, LI, LS e contagem de casos sinalizados (SalePrice e Gr Liv Area)


In [ ]:
# TODO (Acompanhamento 2): boxplots em painel único + investigação dos casos sinalizados


#### 4.1.3 Característica qualitativa B — agrupamento final

In [ ]:
# TODO (Acompanhamento 2/final): frequências, categorias raras, ausências, agrupamento justificado e rótulos finais de B


### 4.2 Análise bivariada

#### 4.2.1 Preço de venda e ano da venda

In [ ]:
# TODO (Acompanhamento final): tabela anual (contagem, mediana) e gráfico de linha da mediana de SalePrice por Yr Sold


#### 4.2.2 Preço de venda e área construída

In [ ]:
# TODO (Acompanhamento final): diagrama de dispersão SalePrice x Gr Liv Area e correlação de Pearson


#### 4.2.3 Preço de venda por categoria de B

In [ ]:
# TODO (Acompanhamento final): tabela (n, mediana, Q1, Q3, IQR) e boxplots de SalePrice por categoria de B


## 5. Exportação dos dados

TODO (Acompanhamento final): registro de nomes originais/utilizados, tratamentos, agrupamento e rótulos finais de B, mapeamento (`SalePrice`, `Gr Liv Area`, `Yr Sold`, `Mo Sold`) e procedimento reproduzível para gerar `data/processed/AmesHousing.csv`.

In [ ]:
# TODO (Acompanhamento final): df.to_csv("../data/processed/AmesHousing.csv", index=False)


## 6. Conclusão e limitações

TODO: natureza observacional, período e local, representatividade, efeito das decisões de preparação, discrepante vs. erro, associação vs. causalidade.

## 7. Contribuições dos integrantes

TODO: descrever a contribuição de cada integrante.

## 8. Declaração de uso de inteligência artificial

TODO: ferramenta/modelo, finalidade, etapas afetadas e verificação do conteúdo (ou declaração de não uso).

## 9. Referências

DE COCK, Dean. Ames, Iowa: alternative to the Boston housing data as an end of semester regression project. *Journal of Statistics Education*, v. 19, n. 3, 2011. Disponível em: <https://jse.amstat.org/v19n3/decock.pdf>.

DE COCK, Dean. *Ames Housing dataset — data documentation* (`DataDocumentation.txt`). Dicionário oficial de variáveis e códigos, fornecido junto ao arquivo bruto (`data/raw/DataDocumentation.txt`). Consultado para interpretar os códigos de variáveis ordinais/nominais, incluindo `KitchenQual` (característica $B$).